# Preparation: Baseline vs. Pipeline A vs. Pipeline B

Pipeline B is represented by the GEPA-optimized run (`GEPA_Qwen3-VL-32B-Thinking.csv`).

All three extraction CSVs already exist, so no `flattening.py` is needed here.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# No flattening needed - all three extraction CSVs already exist.
# They are produced by:
#   ../Baseline-PipelineA/flattening.py  -> baseline.csv, pipelineA.csv
#   ../PipelineB/flattening.py           -> PipelineB-Answers/*.csv

## Import slimmed down Gold-Standard

In [3]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import the Extractions

In [ ]:
baseline_df  = pd.read_csv(Path("../baseline/baseline.csv"))
#pipelineA_df = pd.read_csv(Path("../PipelineA/pipelineA.csv"))
pipelineA_df = pd.read_csv(Path("./../PipelineA/PipelineA-Answers-Thinking-SysP.csv"))
pipelineB_df = pd.read_csv(Path("../PipelineB/PipelineB-Answers/GEPA_Qwen3-VL-32B-Thinking.csv"))

df_to_merge = [
    (baseline_df, "_Base"),
    (pipelineA_df, "_A"),
    (pipelineB_df, "_B"),
]

FileNotFoundError: [Errno 2] No such file or directory: '../Baseline-PipelineA/baseline.csv'

In [ ]:
# Year normalization RegEx-Script

import re

def normalize_year(raw: str, years_present: set[int] | None = None) -> tuple[int | None, str]:
    """Map a raw fiscal-year label to a calendar year. Returns (year, rule_applied)."""

    if isinstance(raw, float):
        if pd.isna(raw):
            return None, "unparseable"
        raw = int(raw) if raw.is_integer() else raw

    label = str(raw).strip().upper().removeprefix("FY").strip()

    if re.fullmatch(r"\d{4}", label):
        return int(label), "plain"
    if re.fullmatch(r"\d{2}", label):
        return 2000 + int(label), "fy_2digit"

    m = re.fullmatch(r"(\d{4})/(\d{1,4})", label)
    if m:
        left, right = m.groups()
        if len(right) == 4:
            return int(left), "range_former_year"  # 2021/2022 -> 2021
        candidates = {int(left), int(left) + 1}
        if years_present:
            hit = candidates & years_present
            if len(hit) == 1:
                return hit.pop(), "fy_end_month_context"
        return int(left), "fy_end_month_fallback"

    return None, "unparseable"

In [ ]:
# Get years from extractions
years_in_extractions = set()
for df, _ in df_to_merge:
    years_in_extractions.update(df["year"].unique().tolist())

# Create ynorm DataFrames to preserve the original ones
baseline_ynorm  = baseline_df.copy()
pipelineA_ynorm = pipelineA_df.copy()
pipelineB_ynorm = pipelineB_df.copy()

df_to_merge_ynorm = [
    (baseline_ynorm, "_Base"),
    (pipelineA_ynorm, "_A"),
    (pipelineB_ynorm, "_B"),
]

# Now normalization via the RegEx Script from above
for df, _ in df_to_merge_ynorm:
    df["year"], df["year_rule"] = zip(*df["year"].apply(
        normalize_year,
        years_present=years_in_extractions
    ))

## Sanity check: report names

In [ ]:
all_extracted = set()
for df, _ in df_to_merge:
    all_extracted.update(df["report_name"].unique())

in_ext_not_gs = sorted(all_extracted - set(gs["report_name"]))
in_gs_not_ext = sorted(set(gs["report_name"]) - all_extracted)

print(f"In extractions, nicht in GS: {len(in_ext_not_gs)} ==> {"OK" if len(in_ext_not_gs) == 0 else "NOK"}")
for r in in_ext_not_gs: print(" ", r)

print(f"\nIn GS, nicht in extractions: {len(in_gs_not_ext)} ==> {"OK" if len(in_gs_not_ext) == 0 else "NOK"}")
for r in in_gs_not_ext: print(" ", r)

## Matching Extractions to Gold-Standard Scheme

### Before ynorm

In [ ]:
years = set()

for df, _ in df_to_merge:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements

print(sorted(years))

### After ynorm

In [ ]:
years = set()

for df, _ in df_to_merge_ynorm:
    years.update(df["year"].unique().tolist()) # update() = add() but for all elements

print(sorted(years, key=lambda y: (y is None, y)))

## Mapping special occurences

In [ ]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)

    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. urspruengliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")

In [ ]:


# Prints out only if sth. goes wrong
for df, sf in df_to_merge_ynorm:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)

    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Alle Werte, die nicht konvertierbar sind (inkl. urspruengliche NaNs)
all_failed = df['value'][converted.isna()]
print(f"Total NaN nach Konversion: {all_failed.notna().sum()}")

## Merging Extraction Values and Gold-Standard

In [ ]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [ ]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")

merged.info()

In [ ]:
merged_ynorm = gs.copy()
merged_ynorm["year"] = merged_ynorm["year"].astype(int)

for df, sf in df_to_merge_ynorm:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged_ynorm = pd.merge(merged_ynorm, agg, on=merge_on, how="left")

merged_ynorm.info()

#### Rows where no value is present must be from Type "NaN" for better analysis; not "None" because the object is missing.

In [ ]:
pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

for col in pipeline_cols:
    merged[col] = merged[col].apply(
    lambda x: np.nan if x is None else x
)

for col in pipeline_cols:
    merged_ynorm[col] = merged_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

### Rearranging Columns

In [ ]:

gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]
merged_ynorm = merged_ynorm[gs_cols + pipeline_cols]

merged.info()

## Saving created DataFrame

In [ ]:
#merged.to_json("Baseline-PipelineA-PipelineB.json", index=False, orient="records", indent=4)
merged_ynorm.to_json("Baseline-PipelineA-PipelineB_ynorm.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("...csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("...json", orient="records")  # Lists instant usable